In [ ]:
# ============================================================
# Project  : Banking Fraud Transactions Analysis
# Author   : Supriya
# Tool     : Python — Pandas only
# Dataset  : Financial Transactions Fraud Dataset
#            Source: Kaggle
# Scale    : 13M+ rows | 2000 Customers | 2010–2019
# ============================================================

In [52]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [53]:
# ══════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════

df = pd.read_csv(r"C:\Banking_Fraud\transactions_data.csv",
                 parse_dates=["date"])

df.shape

(13305915, 12)

In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype         
---  ------          -----         
 0   id              int64         
 1   date            datetime64[us]
 2   client_id       int64         
 3   card_id         int64         
 4   amount          str           
 5   use_chip        str           
 6   merchant_id     int64         
 7   merchant_city   str           
 8   merchant_state  str           
 9   zip             float64       
 10  mcc             int64         
 11  errors          str           
dtypes: datetime64[us](1), float64(1), int64(5), str(5)
memory usage: 1.2 GB


In [55]:
df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [59]:
# ══════════════════════════════════════════════════════════════
# 2. CLEAN TRANSACTIONS
# ══════════════════════════════════════════════════════════════

# clean amount column
df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

# missing value handling
df["merchant_state"].fillna(
    "ONLINE",
    inplace=True
)

df["errors"].fillna(
    "No Errors",
    inplace=True
)

# online transactions have no zip code
df.loc[
    df["merchant_city"] == "ONLINE",
    "zip"
] = 0

# remaining missing zip codes
df["zip"].fillna(
    0,
    inplace=True
)

# validation
df.isna().sum()


id                       0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state     1563700
zip                  89006
mcc                      0
errors            13094522
dtype: int64

In [60]:
# ══════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING — TRANSACTIONS
# ══════════════════════════════════════════════════════════════

df["year"]          = df["date"].dt.year
df["month"]         = df["date"].dt.month
df["hour"]          = df["date"].dt.hour
df["is_online"]     = (df["merchant_city"] == "ONLINE").astype(int)
df["is_refund"]     = (df["amount"] < 0).astype(int)
df["is_high_value"] = (df["amount"] > 500).astype(int)

df[["year", "month", "hour", "is_online", "is_refund", "is_high_value"]].head()

,year,month,hour,is_online,is_refund,is_high_value
0,2010,1,0,0,1,0
1,2010,1,0,0,0,0
2,2010,1,0,0,0,0
3,2010,1,0,0,0,0
4,2010,1,0,0,0,0


In [61]:
# ══════════════════════════════════════════════════════════════
# 4. TRANSACTION ANALYSIS
# ══════════════════════════════════════════════════════════════

df.groupby("use_chip")["amount"].agg(
    transaction_count="count",
    avg_amount="mean",
    total_volume="sum"
).sort_values("total_volume", ascending=False)

,transaction_count,avg_amount,total_volume
use_chip,,,
Swipe Transaction,6967185,41.214899,2.871518e+08
Chip Transaction,4780818,40.889150,1.954836e+08
Online Transaction,1557912,57.256193,8.920011e+07


In [62]:
df.groupby("year")["amount"].agg(
    count="count",
    total="sum",
    avg="mean"
)

,count,total,avg
year,,,
2010,1240880,54232556.12,43.704916
2011,1290770,55778904.96,43.213667
2012,1321672,56832410.86,43.000390
2013,1352808,58284939.62,43.084414
2014,1365537,58617820.51,42.926571
2015,1388065,59514007.43,42.875519
2016,1392117,59844028.90,42.987787
2017,1399308,59628480.63,42.612835
2018,1394792,59627317.94,42.749971


In [63]:
df.groupby("hour")["amount"].count().sort_values(ascending=False).head(5)


hour
12    953498
11    943671
7     901756
13    900703
14    887776
Name: amount, dtype: int64

In [64]:
df.groupby("merchant_state")["amount"].agg(
    count="count",
    total="sum"
).sort_values("total", ascending=False).head(10)

,count,total
merchant_state,,
CA,1427087,59084616.19
TX,1010207,42477208.43
NY,857510,39721647.77
FL,701623,28603402.23
IL,467931,19500277.78
PA,417766,18234988.56
NC,429427,16790704.15
OH,484122,16225273.04
MI,397970,15051464.03


In [65]:
df.groupby("merchant_city")["amount"].agg(
    count="count",
    total="sum"
).sort_values("total", ascending=False).head(10)

,count,total
merchant_city,,
ONLINE,1563700,88896665.09
Houston,146917,6615630.78
Miami,87388,3553176.82
Louisville,66088,3196075.85
San Francisco,47544,3184666.93
Atlanta,54256,3127970.85
Memphis,42154,3080698.92
Las Vegas,51972,2975595.82
Los Angeles,82004,2965266.39


In [66]:
## ══════════════════════════════════════════════════════════════
# 5. CUSTOMER ANALYSIS
# ══════════════════════════════════════════════════════════════

customer_summary = df.groupby("client_id")["amount"].agg(
    total_spent="sum",
    avg_transaction="mean",
    transaction_count="count"
)

customer_summary.sort_values("total_spent", ascending=False).head()

,total_spent,avg_transaction,transaction_count
client_id,,,
96,2445773.25,63.334108,38617
1686,2167880.90,109.433665,19810
1340,2039921.23,92.626855,22023
840,1956340.84,129.601911,15095
464,1882901.35,68.174132,27619


In [67]:
customer_summary.sort_values("transaction_count", ascending=False).head()

,total_spent,avg_transaction,transaction_count
client_id,,,
1098,1460166.80,30.119573,48479
909,927496.96,21.380258,43381
1963,1253911.46,29.530203,42462
1776,1331738.89,32.206503,41350
114,942205.88,23.387923,40286


In [68]:
# ══════════════════════════════════════════════════════════════
# 6. OUTLIER ANALYSIS (IQR)
# ══════════════════════════════════════════════════════════════

Q1, Q3 = df["amount"].quantile(0.25), df["amount"].quantile(0.75)
IQR = Q3 - Q1
LB  = Q1 - 1.5 * IQR
UB  = Q3 + 1.5 * IQR

df[df["amount"] < LB].shape

(386923, 18)

In [69]:
df[df["amount"] > UB].shape

(665596, 18)

In [70]:
# ══════════════════════════════════════════════════════════════
# 7. ERROR ANALYSIS
# ══════════════════════════════════════════════════════════════

df[df["errors"] != "No Errors"].groupby("errors")["amount"].agg(
    count="count",
    avg_amount="mean"
).sort_values("count", ascending=False)

,count,avg_amount
errors,,
Insufficient Balance,130902,68.066661
Bad PIN,32119,39.872121
Technical Glitch,26271,45.051545
Bad Card Number,7767,59.252881
Bad Expiration,6161,58.693889
Bad CVV,6106,61.804916
Bad Zipcode,1126,33.838135
"Bad PIN,Insufficient Balance",293,68.402287
"Insufficient Balance,Technical Glitch",243,76.926584


In [71]:
# ══════════════════════════════════════════════════════════════
# 8. LOAD USERS DATA
# ══════════════════════════════════════════════════════════════

df1 = pd.read_csv(r"C:\Banking_Fraud\users_data.csv")

df1.shape

(2000, 14)

In [72]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   str    
 6   address            2000 non-null   str    
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   str    
 10  yearly_income      2000 non-null   str    
 11  total_debt         2000 non-null   str    
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 218.9 KB


In [73]:
df1.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [74]:
for col in ["per_capita_income", "yearly_income", "total_debt"]:
    df1[col] = df1[col].str.replace("$", "", regex=False).astype(float)

df1.isna().sum()


id                   0
current_age          0
retirement_age       0
birth_year           0
birth_month          0
gender               0
address              0
latitude             0
longitude            0
per_capita_income    0
yearly_income        0
total_debt           0
credit_score         0
num_credit_cards     0
dtype: int64

In [75]:
df1.duplicated().sum()

np.int64(0)

In [76]:
# ══════════════════════════════════════════════════════════════
# 9. FEATURE ENGINEERING — USERS
# ══════════════════════════════════════════════════════════════

df1["years_to_retirement"] = (df1["retirement_age"] - df1["current_age"]).clip(lower=0)
df1["is_retired"]          = (df1["current_age"] >= df1["retirement_age"]).astype(int)
df1["debt_to_income"]      = (df1["total_debt"] / df1["yearly_income"]).round(2)

df1[["years_to_retirement", "is_retired", "debt_to_income"]].head()

,years_to_retirement,is_retired,debt_to_income
0,13,0,2.14
1,15,0,2.48
2,0,1,0.01
3,0,1,0.81
4,27,0,1.68


In [77]:
# ══════════════════════════════════════════════════════════════
# 10. USER ANALYSIS
# ══════════════════════════════════════════════════════════════

df1.describe()

,id,current_age,retirement_age,birth_year,birth_month,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,years_to_retirement,is_retired,debt_to_income
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,999.500000,45.391500,66.237500,1973.803000,6.439000,37.389225,-91.554765,23141.928000,45715.882000,63709.694000,709.734500,3.073000,22.444000,0.151000,1.381665
std,577.494589,18.414092,3.628867,18.421234,3.565338,5.114324,16.283293,11324.137358,22992.615456,52254.453421,67.221949,1.637379,15.988955,0.358138,0.869268
min,0.000000,18.000000,50.000000,1918.000000,1.000000,20.880000,-159.410000,0.000000,1.000000,0.000000,480.000000,1.000000,0.000000,0.000000,0.000000
25%,499.750000,30.000000,65.000000,1961.000000,3.000000,33.837500,-97.395000,16824.500000,32818.500000,23986.750000,681.000000,2.000000,8.000000,0.000000,0.690000
50%,999.500000,44.000000,66.000000,1975.000000,7.000000,38.250000,-86.440000,20581.000000,40744.500000,58251.000000,711.500000,3.000000,22.000000,0.000000,1.420000
75%,1499.250000,58.000000,68.000000,1989.000000,10.000000,41.200000,-80.130000,26286.000000,52698.500000,89070.500000,753.000000,4.000000,36.000000,0.000000,1.960000
max,1999.000000,101.000000,79.000000,2002.000000,12.000000,61.200000,-68.670000,163145.000000,307018.000000,516263.000000,850.000000,9.000000,57.000000,1.000000,4.980000


In [78]:
df1["gender"].value_counts()

gender
Female    1016
Male       984
Name: count, dtype: int64

In [79]:
df1.groupby("gender")[["per_capita_income", "yearly_income",
                        "total_debt", "debt_to_income"]].mean()

,per_capita_income,yearly_income,total_debt,debt_to_income
gender,,,,
Female,23397.224409,46048.314961,63318.339567,1.359961
Male,22878.329268,45372.638211,64113.775407,1.404075


In [80]:
df1["credit_score"].describe()

count    2000.000000
mean      709.734500
std        67.221949
min       480.000000
25%       681.000000
50%       711.500000
75%       753.000000
max       850.000000
Name: credit_score, dtype: float64

In [81]:
pd.cut(df1["credit_score"],
       bins=[300, 579, 669, 739, 799, 850],
       labels=["Poor", "Fair", "Good", "Very Good", "Exceptional"]
).value_counts().sort_index()

credit_score
Poor            81
Fair           348
Good           931
Very Good      474
Exceptional    166
Name: count, dtype: int64

In [82]:
df1.groupby("is_retired")[["yearly_income", "total_debt", "credit_score"]].mean()

,yearly_income,total_debt,credit_score
is_retired,,,
0,47166.739694,72756.783863,710.795053
1,37558.410596,12842.281457,703.771523


In [83]:
df1[["per_capita_income", "yearly_income", "total_debt", "credit_score"]].corr()

,per_capita_income,yearly_income,total_debt,credit_score
per_capita_income,1.000000,0.963975,0.496138,-0.004545
yearly_income,0.963975,1.000000,0.550641,0.000167
total_debt,0.496138,0.550641,1.000000,-0.104537
credit_score,-0.004545,0.000167,-0.104537,1.000000


In [84]:
Q1, Q3 = df1["yearly_income"].quantile(0.25), df1["yearly_income"].quantile(0.75)
IQR = Q3 - Q1

df1[df1["yearly_income"] > Q3 + 1.5 * IQR]

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,years_to_retirement,is_retired,debt_to_income
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4,0,1,0.81
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1,27,0,1.68
21,777,18,65,2002,1,Male,970 Essex Drive,37.37,-122.21,106305.0,216740.0,0.0,700,2,47,0,0.00
40,811,91,68,1929,2,Female,5492 Maple Drive,38.90,-94.68,51642.0,84694.0,2149.0,741,7,0,1,0.03
58,1452,46,59,1973,5,Female,524 Ocean Drive,29.76,-95.38,95039.0,193773.0,241571.0,660,1,13,0,1.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1888,1168,51,68,1968,10,Male,207 Ocean View Street,40.67,-74.42,53790.0,109673.0,242379.0,505,1,17,0,2.21
1924,1790,21,69,1998,3,Male,727 Valley Stream Boulevard,41.24,-73.31,55814.0,113797.0,169684.0,660,1,48,0,1.49
1952,1395,58,65,1961,9,Male,2687 Burns Avenue,40.98,-74.11,75378.0,153691.0,197377.0,604,2,7,0,1.28
1965,628,57,66,1963,1,Male,4 George Lane,40.00,-75.26,52517.0,107075.0,75999.0,815,3,9,0,0.71


In [85]:
df1[df1["yearly_income"] < Q1 - 1.5 * IQR]

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,years_to_retirement,is_retired,debt_to_income
246,1828,50,68,1969,6,Male,35 Hillside Drive,41.83,-87.68,0.0,2466.0,5521.0,711,2,18,0,2.24
662,1016,41,53,1978,8,Female,165 Fourth Street,32.95,-117.19,0.0,553.0,740.0,719,3,12,0,1.34
741,82,22,68,1997,9,Female,6054 Main Avenue,41.83,-87.68,0.0,2026.0,1417.0,779,1,46,0,0.70
751,1151,53,65,1966,12,Male,353 South Boulevard,40.35,-74.65,0.0,920.0,1914.0,788,3,12,0,2.08
764,1942,77,63,1942,3,Female,2073 South Avenue,40.71,-73.99,0.0,1.0,0.0,673,3,0,1,0.00
993,1481,72,65,1947,6,Male,527 Federal Avenue,37.78,-121.99,0.0,2422.0,810.0,632,2,0,1,0.33
1068,1663,24,59,1995,7,Male,276 12th Boulevard,41.83,-87.68,0.0,2370.0,4397.0,581,1,35,0,1.86
1100,1530,64,68,1956,2,Female,25 Elm Street,32.93,-97.22,0.0,1785.0,2892.0,732,3,4,0,1.62
1166,1985,50,69,1969,3,Female,9061 Grant Avenue,34.06,-84.27,0.0,1426.0,3154.0,680,3,19,0,2.21
1213,608,32,65,1987,10,Male,702 Grant Drive,37.78,-121.99,0.0,2365.0,0.0,769,3,33,0,0.00


In [86]:
# ══════════════════════════════════════════════════════════════
# 12. FEATURE ENGINEERING — CARDS
# ══════════════════════════════════════════════════════════════

df2["account_duration_days"] = (df2["expires"] - df2["acct_open_date"]).dt.days
df2["is_chip_enabled"]       = (df2["has_chip"] == "YES").astype(int)

df2[["account_duration_days", "is_chip_enabled"]].head()

,account_duration_days,is_chip_enabled
0,7396,1
1,2436,1
2,7520,1
3,7883,0
4,181,1


In [87]:
# ══════════════════════════════════════════════════════════════
# 15. FRAUD LABELS
# ══════════════════════════════════════════════════════════════

df4 = pd.read_json(r"C:\Banking_Fraud\train_fraud_labels.json")
df4.reset_index(inplace=True)
df4.columns = ["id", "Target"]

df4["Target"].value_counts()

Target
No     8901631
Yes      13332
Name: count, dtype: int64

In [88]:
df4["Target"].value_counts(normalize=True)

Target
No     0.998505
Yes    0.001495
Name: proportion, dtype: float64